# Train a Kamba voice from a contributor's recordings (Understand Kamba)

Fine-tunes Meta's small **MMS Swahili voice** (`facebook/mms-tts-swh`) on recordings of one consenting Kikamba speaker,
so it speaks in her voice and learns Kikamba pronunciation. The result runs fast on an ordinary **CPU**, so it never
depends on a GPU allowance. About **20–30 minutes** on a free T4.

**This is an experiment:** her current recordings are only ~3 minutes (20 clips). The tool's authors recommend
80–150 recordings, so expect a voice that sounds like her but may be rough. The last cells compare it with the
original Swahili voice by ear and with an automatic intelligibility test.

**Before running (Kaggle):**
1. Right panel → *Session options*: **Accelerator: GPU T4 x2**, **Internet: On**.
2. Menu **Add-ons → Secrets → Add Secret**: label `HF_TOKEN`, value = a Hugging Face token that can **read** your
   private datasets (a write token also works). Tick the box to attach it to this notebook.
3. **Run All**. Her recordings stay private: they are read from your private dataset with your token.

In [ ]:
# 1. Setup (~3 min): check GPU, internet and token; install the voice-training tool
import os, socket, subprocess, torch
assert torch.cuda.is_available(), "No GPU: Session options -> Accelerator -> GPU T4 x2"
try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet: Session options -> Internet -> On")

token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not token:
    raise SystemExit("Add your Hugging Face token: Add-ons -> Secrets -> HF_TOKEN (and attach it to this notebook)")
from huggingface_hub import login
login(token=token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = token

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.chdir(WORK)
if not os.path.isdir("finetune-hf-vits"):
    !git clone -q https://github.com/ylacombe/finetune-hf-vits.git
os.chdir(f"{WORK}/finetune-hf-vits")
!git checkout -q 6f3f51f4d667f5c3eef89484d151ffd39d2c2b89
# Versions this tool was tested with (it needs pieces newer transformers/matplotlib removed).
!pip install -q "transformers==4.45.2" "datasets[audio]==3.6.0" "accelerate==1.1.1" "matplotlib==3.9.4" tensorboard Cython soundfile librosa
# The alignment search is compiled C code (the tool's README requires building it).
!cd monotonic_align && mkdir -p monotonic_align && python setup.py build_ext --inplace -q
# One-speaker data has no speaker ids; let the script cope with that.
!sed -i 's/speaker_id=batch\["speaker_id"\]/speaker_id=batch.get("speaker_id")/g' run_vits_finetuning.py
print("Ready:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Starting point (~2 min): the MMS Swahili voice plus the discriminator needed to train it
if not os.path.isdir("mms-swh-train"):
    !python convert_original_discriminator_checkpoint.py --language_code swh --pytorch_dump_folder_path mms-swh-train
print(sorted(os.listdir("mms-swh-train")))

In [ ]:
# 3. Train (~15-25 min). Data: lawmaluki/kamba-voice-tts-contributor (private; 20 clips, text written the
#    way this voice reads it: lower case, ĩ/ũ as i/u). Progress bars show steps and losses.
import json
config = {
    "project_name": "kamba_contributor_voice",
    "push_to_hub": False,
    "report_to": ["tensorboard"],
    "overwrite_output_dir": True,
    "output_dir": f"{WORK}/kamba-voice-ft",
    "dataset_name": "lawmaluki/kamba-voice-tts-contributor",
    "audio_column_name": "audio",
    "text_column_name": "text",
    "train_split_name": "train",
    "eval_split_name": "train",
    "full_generation_sample_text": "wioo museo musyi waku uilyi ata",
    "max_duration_in_seconds": 20,
    "min_duration_in_seconds": 1.0,
    "max_tokens_length": 500,
    "model_name_or_path": "mms-swh-train",
    "preprocessing_num_workers": 2,
    "do_train": True,
    "num_train_epochs": 300,
    "gradient_accumulation_steps": 1,
    "per_device_train_batch_size": 8,
    "learning_rate": 2e-5,
    "adam_beta1": 0.8,
    "adam_beta2": 0.99,
    "warmup_ratio": 0.01,
    "do_eval": True,
    "eval_steps": 150,
    "per_device_eval_batch_size": 4,
    "max_eval_samples": 4,
    "do_step_schedule_per_epoch": True,
    "weight_disc": 3, "weight_fmaps": 1, "weight_gen": 1, "weight_kl": 1.5, "weight_duration": 1, "weight_mel": 35,
    "fp16": True,
    "seed": 456,
}
json.dump(config, open("kamba_config.json", "w"), indent=1)
!CUDA_VISIBLE_DEVICES=0 accelerate launch --num_processes 1 --mixed_precision fp16 run_vits_finetuning.py kamba_config.json 2>&1 | grep -vE "it/s\]|s/it\]|INFO\|" | tail -n 25
assert os.path.exists(f"{WORK}/kamba-voice-ft/model.safetensors"), "Training did not finish - see the lines above"
print("Trained voice saved to", f"{WORK}/kamba-voice-ft")

In [ ]:
# 4. Listen: the same sentences read by the original Swahili voice and by her trained voice
import unicodedata, numpy as np, soundfile as sf
from IPython.display import Audio, display, HTML
from transformers import VitsModel, AutoTokenizer

VOCAB = set(" '-012345678_abcdefghijklmnopqrstuvwyz")
def prep(text):  # the same text preparation as the app (app/tts.py) and the training data
    t = "".join(c for c in unicodedata.normalize("NFD", text) if not unicodedata.combining(c)).replace("’", "'").lower()
    return " ".join("".join(c if c in VOCAB else " " for c in t).split())

voices = {"original Swahili voice": "facebook/mms-tts-swh", "her trained voice": f"{WORK}/kamba-voice-ft"}
loaded = {name: (VitsModel.from_pretrained(p).to("cuda").eval(), AutoTokenizer.from_pretrained(p)) for name, p in voices.items()}
def speak(name, text):
    m, t = loaded[name]
    torch.manual_seed(0)
    with torch.inference_mode():
        return m(**t(prep(text), return_tensors="pt").to("cuda")).waveform[0].float().cpu().numpy()

listen = ["Wĩoo mũseo, mũsyĩ waku ũilyĩ ata?", "Nĩ mũvea mũno nũndũ wa ũtethyo waku.",
          "Ũmũnthĩ nĩkwĩthĩwa na mbua mbingĩ kĩsionĩ kya Machakos.", "Twĩ na mũtũkĩ wa kũthi ũthũkũmonĩ kĩla kwakya."]
os.makedirs(f"{WORK}/samples", exist_ok=True)
for i, s in enumerate(listen, 1):
    display(HTML(f"<b>{i}. {s}</b>"))
    for name in voices:
        w = speak(name, s)
        sf.write(f"{WORK}/samples/{i}-{'trained' if 'trained' in name else 'swahili'}.wav", w, 16000)
        display(HTML(name)); display(Audio(w, rate=16000))
print("WAV files also saved in", f"{WORK}/samples (Output tab -> download)")

In [ ]:
# 5. Intelligibility test: 20 Kikamba sentences she did NOT record, read by both voices and transcribed by the
#    app's Kikamba speech recogniser. Lower character error = clearer. (A proxy, not a listening test.)
import urllib.request, csv, io
from datasets import load_dataset
from transformers import AutoModelForCTC, AutoProcessor
recorded = set(load_dataset("lawmaluki/kamba-voice-tts-contributor", split="train")["text_original"])
tsv = urllib.request.urlopen("https://raw.githubusercontent.com/lawmaluki/understand-kamba/main/app/data/kikamba_prompts.tsv").read().decode("utf-8")
prompts = [r["text"] for r in csv.DictReader((l for l in io.StringIO(tsv) if not l.startswith("#")), delimiter="\t")]
test = [p for p in prompts if p not in recorded and 40 <= len(p) <= 120][:20]

asr_id = "lawmaluki/w2v-bert-2.0_kamba"
proc = AutoProcessor.from_pretrained(asr_id); asr = AutoModelForCTC.from_pretrained(asr_id).to("cuda").eval()
def transcribe(w):
    x = proc(w, sampling_rate=16000, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        return proc.decode(asr(**x).logits.argmax(-1)[0])
def edits(a, b):
    d = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        p, d[0] = d[0], i
        for j, cb in enumerate(b, 1):
            p, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, p + (ca != cb))
    return d[-1]
for name in voices:
    err = tot = 0
    for s in test:
        ref, hyp = prep(s), prep(transcribe(speak(name, s)))
        err += edits(ref, hyp); tot += len(ref)
    print(f"{name:24} character error {100 * err / tot:5.1f}%  (on {len(test)} unseen sentences)")
print("Lower is better. If the trained voice is not clearly better, record more sentences before using it.")

## 6. (Optional) Save her trained voice to your Hugging Face account

Only if it sounds good and step 5 shows it is at least as clear as the Swahili voice. Set `UPLOAD = True` and run.
It is saved **privately** as `lawmaluki/mms-tts-kam-contributor`; the app can then use it as its CPU voice.

In [ ]:
UPLOAD = False
if UPLOAD:
    from huggingface_hub import HfApi
    repo = "lawmaluki/mms-tts-kam-contributor"
    HfApi().create_repo(repo, private=True, exist_ok=True)
    HfApi().upload_folder(folder_path=f"{WORK}/kamba-voice-ft", repo_id=repo, ignore_patterns=["runs/*", "checkpoint*"])
    print("uploaded: https://huggingface.co/" + repo)